# Pseudobulk model building — CoGAPS

**Environment:** `clamp-analyses`

Coordinated Gene Activity in Pattern Sets (CoGAPS) on every pseudobulk dataset. CoGAPS requires non-negative input, so the shifted CPM matrix (after gene filtering, before z-scoring) is used. Reads `k.csv` from CLAMP output for nPatterns. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/CoGAPS/`.

## Libraries

In [ ]:
library(data.table)
library(here)
library(CLAMP)
library(CoGAPS)

set.seed(123)

## Configuration

In [ ]:
DATASET      = "PBMC_Perez2022"
N_ITERATIONS = 5000L
OUT_ROOT     = "output/01_model_building/05_pseudobulk"
DATA_DIR     = "data/pseudobulk"

## Build CoGAPS model for each dataset

In [ ]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)

# Load raw counts (CoGAPS needs non-negative CPM, not zscore)
raw    <- fread(file.path(here(), DATA_DIR, DATASET, "bulk_expr.csv"))
genes  <- raw[[1]]; raw[[1]] <- NULL
counts <- as.matrix(raw); storage.mode(counts) <- "numeric"
rownames(counts) <- genes
samples <- colnames(counts)

cpm  <- CLAMP::cpmCLAMP(counts)
prep <- CLAMP::preprocessCLAMP(Y = cpm, mean_cutoff = 0.5, var_cutoff = 0.1)
cpm_filt   <- prep$Y_filtered
norm_genes <- rownames(cpm_filt)
cat(DATASET, "filtered:", nrow(cpm_filt), "genes x", ncol(cpm_filt), "samples\n")

mat <- cpm_filt - min(cpm_filt)
storage.mode(mat) <- "numeric"

# Load k
k <- as.integer(read.csv(file.path(here(), OUT_ROOT, DATASET, "k.csv"))$k[1])
message("  Using k = ", k, " (from preprocess output)")

# Distributed CoGAPS
n_genes <- nrow(mat)
nSets   <- max(ceiling(n_genes / 2500), 2L)
cat("  nPatterns=", k, "  nSets=", nSets, "\n")

params <- CogapsParams(
  nPatterns    = k,
  nIterations  = N_ITERATIONS,
  seed         = 123,
  distributed  = "genome-wide"
)
params <- setDistributedParams(params, nSets = nSets)

message("  Running CoGAPS ...")
cogaps_res <- CoGAPS(mat, params, nThreads = 4, outputFrequency = 10000)

# Extract B and Z
B <- t(cogaps_res@sampleFactors)
rownames(B) <- paste0("LV", seq_len(nrow(B)))
colnames(B) <- samples

Z <- cogaps_res@featureLoadings
rownames(Z) <- norm_genes
colnames(Z) <- paste0("LV", seq_len(ncol(Z)))

model_dir <- file.path(out_dir, "CoGAPS")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(as.data.frame(B), file.path(model_dir, "B.csv"))
write.csv(as.data.frame(Z), file.path(model_dir, "Z.csv"))
saveRDS(cogaps_res, file.path(model_dir, "cogaps_model.rds"))
message("  CoGAPS saved -> ", model_dir)